# LEGO Minifigure Classification — Training Dashboard

Run all cells, then **re-run the last cell** to refresh with latest training data.

In [ ]:
import re, os, glob, time
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from IPython.display import display, HTML
import subprocess

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

BASE_DIR = '/home/test/big_data_assignment_2'
TMP_DIR = '/tmp/claude-1001/-home-test-big-data-assignment-2/cda7ac9a-9687-43d8-81b6-6853babc1bd9/tasks'

# Training log files
LOG_FILES = {
    'Baseline':    os.path.join(TMP_DIR, 'bz13it5xr.output'),
    'Option B':    os.path.join(TMP_DIR, 'bs6t3bcpi.output'),
    'B Improved':  os.path.join(TMP_DIR, 'bonb4zybn.output'),
    'V2':          os.path.join(TMP_DIR, 'bidttxvy4.output'),
}

# V3 HP Tuning files
V3_HPOPT_DIR = os.path.join(BASE_DIR, 'optionB_v3_hpopt_results')
V3_HP_RESULTS = os.path.join(V3_HPOPT_DIR, 'hp_search_results.txt')
V3_CV_RESULTS = os.path.join(V3_HPOPT_DIR, 'cv_results.txt')
V3_FINAL_COMP = os.path.join(V3_HPOPT_DIR, 'final_comparison.txt')

COLORS = {
    'Baseline': '#e74c3c', 'Option B': '#f39c12', 'B Improved': '#2ecc71', 'V2': '#3498db',
    'V3-Focal': '#9b59b6', 'V3-ArcFace': '#8e44ad', 'V3-SupCon': '#2ecc71'
}

EPOCH_RE = re.compile(
    r'Epoch\s+(\d+)/(\d+)\s*\|\s*'
    r'Train Loss:\s*([\d.]+)\s+Acc:\s*([\d.]+)\s*\|\s*'
    r'Val Loss:\s*([\d.]+)\s+Acc:\s*([\d.]+)'
    r'(?:\s+F1:\s*([\d.]+))?'
)

def parse_log(path):
    if not os.path.exists(path): return []
    with open(path) as f: text = f.read()
    results = []
    for m in EPOCH_RE.finditer(text):
        results.append({
            'epoch': int(m.group(1)), 'total': int(m.group(2)),
            'train_loss': float(m.group(3)), 'train_acc': float(m.group(4)),
            'val_loss': float(m.group(5)), 'val_acc': float(m.group(6)),
            'val_f1': float(m.group(7)) if m.group(7) else None,
        })
    return results

def parse_test_metrics(path):
    """Parse final test metrics from a training log."""
    if not os.path.exists(path): return None
    with open(path) as f: text = f.read()
    if 'EVALUATING' not in text: return None
    section = text[text.find('EVALUATING'):]
    metrics = {}
    for line in section.split('\n'):
        line = line.strip()
        for name in ['Accuracy', 'Top-3 Accuracy', 'Top-5 Accuracy', 'Macro F1', 'Weighted F1']:
            if line.startswith(name):
                try: metrics[name] = float(line.split()[-1])
                except: pass
    return metrics if metrics else None

def check_hp_tuning_status():
    """Check V3 HP tuning progress."""
    result = subprocess.run(['ps', 'aux'], capture_output=True, text=True)
    active = sum(1 for line in result.stdout.split('\n') if 'optionB_v3_hpopt' in line and 'grep' not in line)
    
    hp_done = os.path.exists(V3_HP_RESULTS)
    cv_done = os.path.exists(V3_CV_RESULTS)
    final_done = os.path.exists(V3_FINAL_COMP)
    
    return {
        'active_processes': max(0, active - 1),
        'hp_search_done': hp_done,
        'cv_done': cv_done,
        'final_done': final_done,
    }

print('Setup complete. Run the next cell to render the dashboard.')

---
## Dashboard — Re-run this cell to refresh

In [ ]:
# ============================================================
# V3 HP TUNING STATUS (NEW)
# ============================================================
hp_status = check_hp_tuning_status()
timestamp = time.strftime('%Y-%m-%d %H:%M:%S')

html_v3 = f'<h2 style="color:#2ecc71; font-family:monospace; margin-bottom:20px;">V3 HYPERPARAMETER TUNING + 3-FOLD CV</h2>'
html_v3 += f'<div style="background-color:#1a1a2e; padding:15px; border-left:4px solid #2ecc71; margin-bottom:20px; font-family:monospace; font-size:12px;">'

if hp_status['active_processes'] > 0:
    html_v3 += f'<p style="color:#f1c40f;"><strong>🟢 STATUS: ACTIVELY TRAINING</strong></p>'
    html_v3 += f'<p style="color:#aaa;">Active Processes: {hp_status["active_processes"]}</p>'
    if not hp_status['hp_search_done']:
        html_v3 += f'<p style="color:#f1c40f;"><strong>Current Phase: Phase 1 - Optuna Hyperparameter Search</strong></p>'
        html_v3 += f'<p style="color:#aaa;">Testing 30 hyperparameter combinations (10 per variant)</p>'
    elif not hp_status['cv_done']:
        html_v3 += f'<p style="color:#f1c40f;"><strong>Current Phase: Phase 2 - 3-Fold Cross-Validation</strong></p>'
        html_v3 += f'<p style="color:#aaa;">Running 9 complete training runs with best hyperparameters</p>'
    elif not hp_status['final_done']:
        html_v3 += f'<p style="color:#f1c40f;"><strong>Current Phase: Phase 3 - Final Reporting</strong></p>'
else:
    if hp_status['final_done']:
        html_v3 += f'<p style="color:#2ecc71;"><strong>✓ COMPLETE</strong></p>'
    elif hp_status['cv_done']:
        html_v3 += f'<p style="color:#f39c12;"><strong>⏳ Phase 2 Complete, Phase 3 Running...</strong></p>'
    elif hp_status['hp_search_done']:
        html_v3 += f'<p style="color:#f39c12;"><strong>⏳ Phase 1 Complete, Phase 2 Queued...</strong></p>'
    else:
        html_v3 += f'<p style="color:#888;">Idle</p>'

html_v3 += f'<p style="color:#888; margin-top:10px;">Last refresh: {timestamp}</p>'
html_v3 += f'</div>'

display(HTML(html_v3))

# ============================================================
# PARSE ALL LOGS
# ============================================================
all_data = {}
all_test = {}
for name, path in LOG_FILES.items():
    all_data[name] = parse_log(path)
    all_test[name] = parse_test_metrics(path)

# ============================================================
# STATUS TABLE (HTML) - Original models
# ============================================================
html = f'<h3 style="color:#ccc; font-family:monospace;">Original Models Status</h3>'
html += '<table style="border-collapse:collapse; width:100%; font-family:monospace; font-size:13px;">'
html += '<tr style="border-bottom:2px solid #555;">'
html += '<th style="text-align:left;padding:8px;color:#888;">Model</th>'
html += '<th style="text-align:left;padding:8px;color:#888;">Classes</th>'
html += '<th style="text-align:left;padding:8px;color:#888;">Status</th>'
html += '<th style="text-align:left;padding:8px;color:#888;">Best Val Acc</th>'
html += '<th style="text-align:left;padding:8px;color:#888;">Best Val F1</th>'
html += '<th style="text-align:left;padding:8px;color:#888;">Test Acc</th>'
html += '<th style="text-align:left;padding:8px;color:#888;">Test Macro F1</th>'
html += '</tr>'

class_counts = {'Baseline': 122, 'Option B': 28, 'B Improved': 28, 'V2': 37}

for name in ['Baseline', 'Option B', 'B Improved', 'V2']:
    color = COLORS[name]
    epochs = all_data[name]
    test = all_test[name]
    n_cls = class_counts.get(name, '?')
    
    if not epochs:
        status = '<span style="color:#888;">No log</span>'
        best_acc = best_f1 = test_acc = test_f1 = '-'
    else:
        cur, tot = epochs[-1]['epoch'], epochs[-1]['total']
        best_acc = f"{max(e['val_acc'] for e in epochs):.1%}"
        f1s = [e['val_f1'] for e in epochs if e['val_f1'] is not None]
        best_f1 = f"{max(f1s):.4f}" if f1s else '-'
        
        # Check if done
        log_text = open(LOG_FILES[name]).read() if os.path.exists(LOG_FILES[name]) else ''
        is_done = cur >= tot or 'Early stopping' in log_text or 'EVALUATING' in log_text
        
        if is_done:
            status = f'<span style="color:#2ecc71;">&#10004; Done ({cur} ep)</span>'
        else:
            pct = cur / tot * 100
            bar = '█' * int(pct / 5) + '░' * (20 - int(pct / 5))
            status = f'<span style="color:#f1c40f;">⬤ {cur}/{tot} [{bar}] {pct:.0f}%</span>'
        
        test_acc = f"{test['Accuracy']:.1%}" if test and 'Accuracy' in test else '-'
        test_f1 = f"{test['Macro F1']:.4f}" if test and 'Macro F1' in test else '-'
    
    html += f'<tr style="border-bottom:1px solid #333;">'
    html += f'<td style="padding:8px;"><span style="color:{color};font-weight:bold;font-size:14px;">● {name}</span></td>'
    html += f'<td style="padding:8px;color:#aaa;">{n_cls}</td>'
    html += f'<td style="padding:8px;">{status}</td>'
    html += f'<td style="padding:8px;color:white;font-weight:bold;">{best_acc}</td>'
    html += f'<td style="padding:8px;color:white;">{best_f1}</td>'
    html += f'<td style="padding:8px;color:#2ecc71;font-weight:bold;">{test_acc}</td>'
    html += f'<td style="padding:8px;color:#2ecc71;">{test_f1}</td>'
    html += '</tr>'

html += '</table>'
display(HTML(html))

In [ ]:
# ============================================================
# TRAINING CURVES — 2x3 grid
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.patch.set_facecolor('#0f0f23')

titles = ['Train Loss', 'Validation Loss', 'Validation Accuracy',
          'Train Accuracy', 'Overfit Gap (Train-Val Acc)', 'Validation Macro F1']
ylabels = ['Loss', 'Loss', 'Accuracy', 'Accuracy', 'Gap', 'F1']

for ax, title, ylabel in zip(axes.flat, titles, ylabels):
    ax.set_facecolor('#1a1a2e')
    ax.set_title(title, color='white', fontsize=11, fontweight='bold')
    ax.set_xlabel('Epoch', color='#888', fontsize=9)
    ax.set_ylabel(ylabel, color='#888', fontsize=9)
    ax.tick_params(colors='#888', labelsize=8)
    for spine in ax.spines.values(): spine.set_color('#333')
    ax.grid(True, alpha=0.12)

for name in ['Baseline', 'Option B', 'B Improved', 'V2']:
    epochs = all_data[name]
    if not epochs: continue
    color = COLORS[name]
    ep = [e['epoch'] for e in epochs]
    kw = dict(color=color, linewidth=2, label=name, marker='o', markersize=3, alpha=0.9)
    
    axes[0, 0].plot(ep, [e['train_loss'] for e in epochs], **kw)
    axes[0, 1].plot(ep, [e['val_loss'] for e in epochs], **kw)
    axes[0, 2].plot(ep, [e['val_acc'] for e in epochs], **kw)
    axes[1, 0].plot(ep, [e['train_acc'] for e in epochs], **kw)
    axes[1, 1].plot(ep, [e['train_acc'] - e['val_acc'] for e in epochs], **kw)
    
    if epochs[0]['val_f1'] is not None:
        axes[1, 2].plot(ep, [e['val_f1'] for e in epochs], **kw)

axes[1, 1].axhline(y=0, color='white', linewidth=0.5, alpha=0.3)

for ax in axes.flat:
    if ax.get_legend_handles_labels()[1]:
        ax.legend(fontsize=7, facecolor='#1a1a2e', edgecolor='#333', labelcolor='white')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# FINAL TEST METRICS COMPARISON
# ============================================================
final = {}
for name in ['Baseline', 'Option B', 'B Improved', 'V2']:
    t = all_test.get(name)
    if t: final[name] = t

if final:
    fig, ax = plt.subplots(figsize=(16, 6))
    fig.patch.set_facecolor('#0f0f23')
    ax.set_facecolor('#1a1a2e')
    
    models = list(final.keys())
    x = np.arange(len(models))
    metrics = ['Accuracy', 'Top-3 Accuracy', 'Top-5 Accuracy', 'Macro F1', 'Weighted F1']
    short_names = ['Accuracy', 'Top-3', 'Top-5', 'Macro F1', 'Wt F1']
    bar_colors = ['#3498db', '#f39c12', '#9b59b6', '#e74c3c', '#2ecc71']
    w = 0.15
    
    for i, (metric, sname, bc) in enumerate(zip(metrics, short_names, bar_colors)):
        vals = [final[m].get(metric, 0) for m in models]
        bars = ax.bar(x + (i - 2) * w, vals, w, label=sname, color=bc, alpha=0.85)
        for bar in bars:
            h = bar.get_height()
            if h > 0:
                ax.annotate(f'{h:.2f}', xy=(bar.get_x() + w/2, h),
                           xytext=(0, 2), textcoords='offset points',
                           ha='center', va='bottom', fontsize=7, color='white')
    
    ax.set_xticks(x)
    ax.set_xticklabels(models, fontsize=1)
    for i, m in enumerate(models):
        ax.text(i, -0.07, m, ha='center', va='top', fontsize=11,
                color=COLORS.get(m, 'white'), fontweight='bold',
                transform=ax.get_xaxis_transform())
    
    ax.set_ylim(0, 1.1)
    ax.set_title('Final Test Metrics — All Models', color='white', fontsize=13, fontweight='bold')
    ax.tick_params(colors='#888')
    for spine in ax.spines.values(): spine.set_color('#333')
    ax.grid(True, alpha=0.12, axis='y')
    ax.legend(fontsize=9, facecolor='#1a1a2e', edgecolor='#333', labelcolor='white',
             ncol=5, loc='upper left')
    plt.tight_layout()
    plt.show()
else:
    print('No completed test metrics yet.')

In [ ]:
# ============================================================
# PER-EPOCH DETAIL TABLE
# ============================================================
for name in ['Baseline', 'Option B', 'B Improved', 'V2']:
    epochs = all_data[name]
    if not epochs: continue
    
    color = COLORS[name]
    html = f'<h3 style="color:{color};">● {name} — Epoch Details</h3>'
    html += '<table style="border-collapse:collapse; font-family:monospace; font-size:12px; width:100%;">'
    html += '<tr style="border-bottom:2px solid #555;">'
    for col in ['Epoch', 'Train Loss', 'Train Acc', 'Val Loss', 'Val Acc', 'Val F1', 'Overfit Gap']:
        html += f'<th style="text-align:right;padding:4px 8px;color:#888;">{col}</th>'
    html += '</tr>'
    
    best_val_acc = max(e['val_acc'] for e in epochs)
    
    for e in epochs:
        gap = e['train_acc'] - e['val_acc']
        gap_color = '#e74c3c' if gap > 0.2 else '#f39c12' if gap > 0.1 else '#2ecc71'
        is_best = e['val_acc'] == best_val_acc
        row_bg = 'background-color:#1a3a1a;' if is_best else ''
        star = ' ★' if is_best else ''
        f1_str = f"{e['val_f1']:.4f}" if e['val_f1'] is not None else '-'
        
        html += f'<tr style="border-bottom:1px solid #222;{row_bg}">'
        html += f'<td style="text-align:right;padding:4px 8px;color:#ccc;">{e["epoch"]}/{e["total"]}{star}</td>'
        html += f'<td style="text-align:right;padding:4px 8px;color:#ccc;">{e["train_loss"]:.4f}</td>'
        html += f'<td style="text-align:right;padding:4px 8px;color:#ccc;">{e["train_acc"]:.1%}</td>'
        html += f'<td style="text-align:right;padding:4px 8px;color:#ccc;">{e["val_loss"]:.4f}</td>'
        html += f'<td style="text-align:right;padding:4px 8px;color:white;font-weight:bold;">{e["val_acc"]:.1%}</td>'
        html += f'<td style="text-align:right;padding:4px 8px;color:#ccc;">{f1_str}</td>'
        html += f'<td style="text-align:right;padding:4px 8px;color:{gap_color};">{gap:+.1%}</td>'
        html += '</tr>'
    
    html += '</table><br>'
    display(HTML(html))

In [ ]:
# ============================================================
# IMPROVEMENT WATERFALL CHART
# ============================================================
stages = []
acc_values = []

# Build stages from available test results
if all_test.get('Baseline'):
    stages.append('Baseline\n(122 cls)')
    acc_values.append(all_test['Baseline'].get('Accuracy', 0))
if all_test.get('Option B'):
    stages.append('+ Category\nMerge (28 cls)')
    acc_values.append(all_test['Option B'].get('Accuracy', 0))
if all_test.get('B Improved'):
    stages.append('+ Augmentation\n+ Focal Loss\n+ Larger imgs')
    acc_values.append(all_test['B Improved'].get('Accuracy', 0))
if all_test.get('V2'):
    stages.append('+ B2 backbone\n+ BatchNorm\n+ Town split\n+ TTA')
    acc_values.append(all_test['V2'].get('Accuracy', 0))

if len(stages) >= 2:
    fig, ax = plt.subplots(figsize=(14, 6))
    fig.patch.set_facecolor('#0f0f23')
    ax.set_facecolor('#1a1a2e')
    
    x = np.arange(len(stages))
    bar_colors = ['#e74c3c', '#f39c12', '#2ecc71', '#3498db'][:len(stages)]
    
    bars = ax.bar(x, acc_values, 0.6, color=bar_colors, alpha=0.85, edgecolor='white', linewidth=0.5)
    
    # Add value labels and delta arrows
    for i, (bar, val) in enumerate(zip(bars, acc_values)):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.01, f'{val:.1%}',
               ha='center', va='bottom', fontsize=12, color='white', fontweight='bold')
        if i > 0:
            delta = val - acc_values[i-1]
            ax.annotate(f'+{delta:.1%}', xy=(i, acc_values[i-1] + delta/2),
                       fontsize=10, color='#f1c40f', fontweight='bold', ha='center')
    
    ax.set_xticks(x)
    ax.set_xticklabels(stages, color='#ccc', fontsize=9)
    ax.set_ylim(0, 1.05)
    ax.set_title('Accuracy Improvement Waterfall', color='white', fontsize=13, fontweight='bold')
    ax.set_ylabel('Test Accuracy', color='#888')
    ax.tick_params(colors='#888')
    for spine in ax.spines.values(): spine.set_color('#333')
    ax.grid(True, alpha=0.12, axis='y')
    plt.tight_layout()
    plt.show()
else:
    print('Need at least 2 completed models for waterfall chart.')

In [ ]:
# ============================================================
# LEARNING RATE SCHEDULE VISUALIZATION
# ============================================================
fig, axes = plt.subplots(1, len([n for n in all_data if all_data[n]]),
                          figsize=(5 * len([n for n in all_data if all_data[n]]), 4))
fig.patch.set_facecolor('#0f0f23')
if not isinstance(axes, np.ndarray): axes = [axes]

lr_re = re.compile(r'LR:\s*([\d.]+)')

for ax, name in zip(axes, [n for n in ['Baseline', 'Option B', 'B Improved', 'V2'] if all_data.get(n)]):
    color = COLORS[name]
    ax.set_facecolor('#1a1a2e')
    ax.set_title(f'{name} — LR Schedule', color=color, fontsize=10, fontweight='bold')
    ax.tick_params(colors='#888', labelsize=7)
    for spine in ax.spines.values(): spine.set_color('#333')
    ax.grid(True, alpha=0.12)
    ax.set_xlabel('Epoch', color='#888', fontsize=8)
    ax.set_ylabel('Learning Rate', color='#888', fontsize=8)
    
    # Parse LR from log
    if os.path.exists(LOG_FILES[name]):
        with open(LOG_FILES[name]) as f:
            text = f.read()
        lrs = []
        for line in text.split('\n'):
            if 'Epoch' in line and 'LR:' in line:
                m = lr_re.search(line)
                if m: lrs.append(float(m.group(1)))
        if lrs:
            ax.plot(range(1, len(lrs)+1), lrs, color=color, linewidth=2, marker='o', markersize=3)
            ax.set_yscale('log')

plt.tight_layout()
plt.show()

---
### How to refresh
Click **Cell → Run All Below** from the Status Table cell, or press `Shift+Enter` on each cell to re-read the latest training data.

In [ ]:
# ============================================================
# V3 HP TUNING RESULTS (if available)
# ============================================================
if os.path.exists(V3_HP_RESULTS):
    with open(V3_HP_RESULTS) as f:
        hp_content = f.read()
    html_hp = '<h3 style="color:#2ecc71;">Hyperparameter Search Results</h3>'
    html_hp += '<pre style="background-color:#1a1a2e; padding:15px; border-left:4px solid #2ecc71; overflow-x:auto;">'
    html_hp += hp_content.replace('<', '&lt;').replace('>', '&gt;')
    html_hp += '</pre>'
    display(HTML(html_hp))
else:
    display(HTML('<p style="color:#888; font-style:italic;">Hyperparameter search results will appear here when Phase 1 completes...</p>'))

if os.path.exists(V3_CV_RESULTS):
    with open(V3_CV_RESULTS) as f:
        cv_content = f.read()
    html_cv = '<h3 style="color:#3498db;">Cross-Validation Results</h3>'
    html_cv += '<pre style="background-color:#1a1a2e; padding:15px; border-left:4px solid #3498db; overflow-x:auto;">'
    html_cv += cv_content.replace('<', '&lt;').replace('>', '&gt;')
    html_cv += '</pre>'
    display(HTML(html_cv))
else:
    display(HTML('<p style="color:#888; font-style:italic;">Cross-validation results will appear here when Phase 2 completes...</p>'))

if os.path.exists(V3_FINAL_COMP):
    with open(V3_FINAL_COMP) as f:
        final_content = f.read()
    html_final = '<h3 style="color:#f39c12;">Final Comparison vs Baseline V3</h3>'
    html_final += '<pre style="background-color:#1a1a2e; padding:15px; border-left:4px solid #f39c12; overflow-x:auto;">'
    html_final += final_content.replace('<', '&lt;').replace('>', '&gt;')
    html_final += '</pre>'
    display(HTML(html_final))
else:
    display(HTML('<p style="color:#888; font-style:italic;">Final comparison will appear here when Phase 3 completes...</p>'))

---
## V3 Hyperparameter Tuning Results (when available)